# Experiment 16: Spatial Post-Smoothing 🧬
## Fusion of Tree-Based Tabular Superiority & Graph Attention Networks

### 🎯 The Dilemma
Through our previous 5-Fold validation experiments, we proved two things:
1. **Tree-based models (LightGBM/XGBoost)** dominate deep neural networks (MLPs) on the tabular protein expression data, easily achieving 90.27% accuracy.
2. **Graph Neural Networks (GraphSAGE)** applied directly to raw proteins with $K=5$ fail miserably (83.2%), as they struggle to learn the tabular features and have a very small reception field.

### 💡 The "Spatial Post-Smoothing" Concept
Instead of forcing a GNN to figure out complex protein marker patterns, we **offload the tabular interpretation entirely to LightGBM.** 

**The Pipeline:**
1. **Phase 1: Tabular Base (LightGBM)** - Train LightGBM strictly on the protein markers to predict cell classes. Extract the raw multi-class probability outputs (an 18-dim probability vector per cell).
2. **Phase 2: Graph Construction (Macro $K=20$)** - Construct a much larger regional spatial graph using the `X_cent` and `Y_cent` coordinates.
3. **Phase 3: Spatial Smoothing (GATv2)** - We feed **ONLY the predicted node probabilities** into a 1 or 2-layer Graph Attention Network.

In [ ]:
# ==========================================
# 0. Kaggle Environment Setup & Imports
# ==========================================
# Install PyTorch Geometric (uncomment on Kaggle if needed)
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report, accuracy_score

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

import warnings
warnings.filterwarnings('ignore')

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 📥 Step 1: Load and Prepare the CODEX Data
*Note: Update the file paths below to point to the Kaggle dataset paths once uploaded.*

In [ ]:
# ==========================================
# 1. Load Data
# ==========================================
DATA_PATH = '/kaggle/input/datasets/amshahriarrashidmahe/chl-codex-annotated/cHL_CODEX_annotation.csv'

try:
    df = pd.read_csv(DATA_PATH)
    
    # Since you provided a single CSV, let's create a spatial split (train/valid)
    # Alternatively, you can use train_test_split if Random Split is preferred
    # For now, let's simulate a split (e.g., leaving one Region_ID out for validation)
    # Adjust this based on your standard splitting strategy!
    if 'Region_ID' in df.columns:
        regions = df['Region_ID'].unique()
        # Simple hold-out split: keep one region for validation
        valid_region = regions[-1] 
        train_df = df[df['Region_ID'] != valid_region].copy()
        valid_df = df[df['Region_ID'] == valid_region].copy()
        print(f"Validation Region: {valid_region}")
    else:
        from sklearn.model_selection import train_test_split
        train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['cell_label'])
        
except FileNotFoundError:
    print("Files not found. Please verify the Kaggle dataset paths.")
    train_df = None
    valid_df = None

target_col = 'cell_label'
spatial_cols = ['X_cent', 'Y_cent']
ignore_cols = ['cell_id', 'Region_ID'] + spatial_cols + [target_col]

def prepare_xy(df):
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    X_proteins = df[feature_cols].values
    X_spatial = df[spatial_cols].values
    Y_labels = df[target_col].values
    return X_proteins, X_spatial, Y_labels

if train_df is not None:
    X_prot_train, X_spat_train, y_train = prepare_xy(train_df)
    X_prot_valid, X_spat_valid, y_valid = prepare_xy(valid_df)
    
    num_classes = len(np.unique(y_train))
    print(f"Train Shape: {X_prot_train.shape}, Classes: {num_classes}")

### 🌳 Step 2: Tabular Base Predictions (LightGBM)

In [ ]:
# ==========================================
# 2. Phase 1: Train Base Tree Model Models
# ==========================================
print("Training Base LightGBM Model on Protein Features...")

lgb_params = {
    'objective': 'multiclass',
    'num_class': 18, # Update exactly to your actual class count
    'metric': 'multi_error',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'n_estimators': 300,
    'random_state': 42,
    'verbose': -1
}

if train_df is not None:
    lgb_params['num_class'] = num_classes
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_prot_train, y_train)

    # Calculate probabilities (N, num_classes)
    train_probs = model_lgb.predict_proba(X_prot_train)
    valid_probs = model_lgb.predict_proba(X_prot_valid)

    # Initial Baseline evaluation
    valid_preds = np.argmax(valid_probs, axis=1)
    base_f1 = f1_score(y_valid, valid_preds, average='macro')
    print(f"LightGBM Base Validation Macro F1: {base_f1:.4f}")

### 🕸️ Step 3: Construct the Spatial Graph (Macro $K=20$)

In [ ]:
# ==========================================
# 3. Phase 2: Macro Spatial Graph Const.
# ==========================================
K_NEIGHBORS = 20

def construct_graph(probs, coords, labels, k=K_NEIGHBORS):
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
    nbrs.fit(coords)
    distances, indices = nbrs.kneighbors(coords)
    
    # Build edge_index
    source_nodes = np.repeat(np.arange(len(coords)), k)
    target_nodes = indices[:, 1:].flatten()
    
    edge_index = np.vstack((source_nodes, target_nodes))
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    
    # Features are purely the model probabilities from LightGBM
    x = torch.tensor(probs, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)
    
    return Data(x=x, edge_index=edge_index, y=y)

if train_df is not None:
    print(f"Building KNN Graphs with K={K_NEIGHBORS}...")
    train_graph = construct_graph(train_probs, X_spat_train, y_train)
    valid_graph = construct_graph(valid_probs, X_spat_valid, y_valid)
    
    train_graph = train_graph.to(device)
    valid_graph = valid_graph.to(device)
    print("Graph construction complete!")

### 🧠 Step 4: Spatial Post-Smoothing with GATv2

In [ ]:
# ==========================================
# 4. Phase 3: Spatial Smoothing Engine
# ==========================================
class SpatialSmoother(torch.nn.Module):
    def __init__(self, num_classes, hidden_dim=64):
        super(SpatialSmoother, self).__init__()
        self.gat1 = GATv2Conv(num_classes, hidden_dim, heads=4, concat=True, dropout=0.2)
        self.gat2 = GATv2Conv(hidden_dim * 4, num_classes, heads=1, concat=False, dropout=0.2)

    def forward(self, x, edge_index):
        identity = x # skip-connection: preserve LightGBM decision
        
        out = self.gat1(x, edge_index)
        out = F.elu(out)
        out = F.dropout(out, p=0.2, training=self.training)
        
        out = self.gat2(out, edge_index)
        return out + identity # Residual connection to baseline probs

if train_df is not None:
    model_gnn = SpatialSmoother(num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model_gnn.parameters(), lr=0.005, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    def train():
        model_gnn.train()
        optimizer.zero_grad()
        out = model_gnn(train_graph.x, train_graph.edge_index)
        loss = criterion(out, train_graph.y)
        loss.backward()
        optimizer.step()
        return loss.item()

    @torch.no_grad()
    def test(graph):
        model_gnn.eval()
        out = model_gnn(graph.x, graph.edge_index)
        pred = out.argmax(dim=1)
        correct = (pred == graph.y).sum().item()
        acc = correct / len(graph.y)
        f1 = f1_score(graph.y.cpu(), pred.cpu(), average='macro')
        return acc, f1, pred.cpu()

    EPOCHS = 100
    print("Training GNN Spatial Smoother...")
    best_val_f1 = 0
    final_preds = None

    for epoch in range(1, EPOCHS + 1):
        loss = train()
        if epoch % 10 == 0 or epoch == 1:
            train_acc, train_f1, _ = test(train_graph)
            val_acc, val_f1, val_preds = test(valid_graph)
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                final_preds = val_preds
                
            print(f"Epoch {epoch:03d}: Loss: {loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    print("=" * 50)
    print(f"🌲 BASE LIGHTGBM VALIDATION F1     : {base_f1:.4f}")
    print(f"🌐 GATv2 SMOOTHED VALIDATION F1    : {best_val_f1:.4f}")
    print("=" * 50)

### 📊 Step 5: Final Evaluation & Diagnostics

In [ ]:
# ==========================================
# 5. Diagnostic Reporting
# ==========================================
if train_df is not None:
    lgb_preds = np.argmax(valid_probs, axis=1)
    
    changed_preds = (final_preds.numpy() != lgb_preds).sum()
    total_preds = len(lgb_preds)
    
    print(f"\nTotal predictions altered by GNN: {changed_preds} / {total_preds} ({(changed_preds/total_preds)*100:.2f}%)")
    
    print("\n--- Final Performance with Spatial Context ---")
    print(classification_report(y_valid, final_preds))